# Edge-learning readiness review — corrected implementation

This notebook is a read-only status check. It does not create a cohort, write outcomes, start a resolver, or alter active telemetry. The prior review found defects in the earlier prepared shadow module; those defects are covered by the current regression tests.


In [ ]:
from pathlib import Path
from contextlib import closing
from datetime import datetime, timezone
from hashlib import sha256
import sqlite3
import sys

repo = Path.cwd()
if not (repo / 'shadow_cohort.py').exists():
    repo = repo.parent
sys.path.insert(0, str(repo))
import shadow_cohort as shadow

uri = (repo / 'data/telemetry_calibration.sqlite3').resolve().as_uri() + '?mode=ro'
with closing(sqlite3.connect(uri, uri=True)) as connection:
    run_id, started, ends = connection.execute('SELECT run_id, started_ts, planned_end_ts FROM telemetry_runs ORDER BY started_ts DESC LIMIT 1').fetchone()
    rows, first_ts, last_ts = connection.execute('SELECT COUNT(*), MIN(ts), MAX(ts) FROM telemetry_snapshots WHERE run_id=?', (run_id,)).fetchone()

print({
    'checked_at_utc': datetime.now(timezone.utc).isoformat(),
    'shadow_source_sha256': sha256((repo / 'shadow_cohort.py').read_bytes()).hexdigest(),
    'telemetry_run_id': run_id,
    'telemetry_rows': rows,
    'telemetry_hours_observed': (last_ts - first_ts) / 3600,
    'planned_telemetry_end_utc': datetime.fromtimestamp(ends, timezone.utc).isoformat(),
    'outcome_labels_in_telemetry': False,
    'shadow_databases_present': [path.name for path in sorted((repo / 'data').glob('*shadow*.sqlite3'))],
    'live_buy_enabled': False,
})

print({
    'locked_parameters': dict(shadow.LOCKED_PARAMETERS),
    'primary_inference_unit': 'first accepted candidate frozen per UTC-hour episode',
    'report_gates': ['30 calendar days', 'heartbeat evidence with no gap above 300 seconds', '100 resolved candidate rows', '30 resolved accepted representatives'],
    'resolver_states': ['PENDING', 'RESOLVED', 'VOID', 'QUARANTINED', 'RETRY_EXHAUSTED'],
})


## Interpretation

Telemetry has no terminal outcomes, so it cannot train a model or estimate edge. The next cohort must be a future collection with full market state, valid books, a non-future 10-second baseline, market-specific fee evidence and terminal resolution evidence. The report uses net estimated PnL after taker fee, but it remains a shadow estimate until an execution-and-cost study is separately completed.
